In [225]:
# Data
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

# Fuzzy matching
from thefuzz import fuzz, process

# Scikit-learn
from sklearn.model_selection import (
    train_test_split, KFold, StratifiedKFold,
    cross_val_score, cross_validate, GridSearchCV, RandomizedSearchCV
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer

# Models
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, SGDRegressor
)
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor

# Metrics
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    median_absolute_error, mean_absolute_percentage_error,
    root_mean_squared_error, make_scorer
)

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


In [ ]:
# For imputation
# explicitly require this experimental feature
from sklearn.experimental import enable_iterative_imputer  # noqa
# now you can import normally from sklearn.impute
from sklearn.impute import IterativeImputer

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import KNNImputer

In [186]:
def process_dataframe(df):
    """
    Apply string cleaning, brand/model corrections, and fuzzy matching.
    These operations don't require fitting on training data.
    """

    df = df.copy()
    # ============================================================================
    # SECTION 1: REFERENCE DATA SETUP
    # ============================================================================
    
    # Reference list of correct model names
    models = ["golf", "veloste", "caddy", "yaris", "q2", "fiesta", "2 series", "3 series", "a3", "octavia", 
              "passat", "focus", "insignia", "a class", "q3", "fabia", "ka+", "glc class", "i30", "c class", 
              "polo", "e class", "q5", "up", "c-hr", "mokka x", "corsa", "astra", "tt", "5 series", "aygo", 
              "4 series", "slk", "viva", "t-roc", "ecosport", "tucson", "x-class", "cl class", "ix20", "i20", 
              "rapid", "a1", "auris", "sharan", "adam", "x3", "a8", "gls class", "b-max", "a4", "kona", "i10", 
              "mokka", "s-max", "x2", "crossland x", "tiguan", "a5", "gle class", "zafira", "ioniq", "a6", 
              "mondeo", "yeti outdoor", "x1", "scala", "s class", "1 series", "kamiq", "kuga", "tourneo connect", 
              "q7", "gla class", "arteon", "sl class", "santa fe", "grandland x", "i800", "rav4", "touran", 
              "citigo", "roomster", "prius", "corolla", "b class", "kodiaq", "v class", "caddy maxi life", 
              "superb", "getz", "combo life", "beetle", "galaxy", "m3", "gtc", "x4", "ka", "ix35", 
              "grand tourneo connect", "m4", "tourneo custom", "z4", "x5", "meriva", "rs6", "verso", "touareg", 
              "shuttle", "cls class", "c-max", "puma", "cla class", "i40", "tiguan allspace", "6 series", 
              "caravelle", "karoq", "i3", "grand c-max", "t-cross", "a7", "golf sv", "agila", "gt86", "yeti", 
              "california", "land cruiser", "edge", "x6", "caddy life", "8 series", "fusion", "gl class", 
              "scirocco", "z3", "proace verso", "hilux", "amarok", "cc", "7 series", "avensis", "eos", "m class", 
              "grandland", "zafira tourer", "rs5", "r8", "mustang", "antara", "q8", "camry", "clk", "rs3", 
              "jetta", "kadjar", "sq5", "rs4", "supra", "i8", "x7", "sq7", "g class", "s3", "crossland", 
              "tigra", "escort", "glb class", "vivaro", "verso-s", "m5", "s4", "iq", "a2", "caddy maxi", 
              "streetka", "cascada", "accent", "s8", "rs", "golf s", "ranger", "vectra", "ampera", "fox", 
              "urban cruiser", "m2", "clc class", "m6", "s5", "terracan", "200", "220", "230", "NaN"]
    
    # Get unique short model names (2 characters) for separate handling
    short_models = [models[i] for i in range(len(models)) if len(models[i]) == 2]
    short_models = list(set(short_models))
    
    transmission_types = ["semi-auto", "manual", "automatic", "unkown", "NaN", "other"]
    fuel_types = ["petrol", "diesel", "hybrid", "electric", "other", "NaN"]
    
    # Brand name corrections mapping
    brand_mapping = {
        "vw": "vw",
        "v": "vw",
        "w": "vw",
        
        "toyota": "toyota",
        "toyot": "toyota",
        "oyota": "toyota",
        
        "audi": "audi",
        "aud": "audi",
        "udi": "audi",
        "ud": "audi",
        
        "ford": "ford",
        "for": "ford",
        "ord": "ford",
        "or": "ford",
        
        "bmw": "bmw",
        "bm": "bmw",
        "mw": "bmw",
        
        "skoda": "skoda",
        "skod": "skoda",
        "koda": "skoda",
        "kod": "skoda",
        
        "opel": "opel",
        "ope": "opel",
        "pel": "opel",
        "pe": "opel",
        
        "mercedes": "mercedes",
        "mercede": "mercedes",
        "ercedes": "mercedes",
        "ercede": "mercedes",
        
        "hyundai": "hyundai",
        "hyunda": "hyundai",
        "yundai": "hyundai",
        "yunda": "hyundai"
    }
    
    # ============================================================================
    # SECTION 2: INITIAL CLEANING (NO FITTING REQUIRED) -> no risk of data leakage
    # ============================================================================
        
    # Convert brand names to lowercase and removes all beginning and trailing whitespace (e.g. space) from the column
    df["Brand"] = df["Brand"].str.lower().str.strip()
    df["model"] = df["model"].str.lower().str.strip()
    df["transmission"] = df["transmission"].str.lower().str.strip()
    df["fuelType"] = df["fuelType"].str.lower().str.strip()
    
    # Replace NaN with string NaN to avoid errors in the fuzzy algorithm (cant match NaNs)
    df[["model", "transmission", "fuelType"]] = df[["model", "transmission", "fuelType"]].fillna("NaN")
    
    # 1.1 Fixing brands
    df["Brand"] = df["Brand"].map(brand_mapping)
    
    # 1.2 Fixing models
    # Similarity matching for Models, Transmission and fuel columns (Fuzzy Match)
    # Source for process.extractOne (fuzzy): https://github.com/seatgeek/thefuzz
    
    # VECTORIZED APPROACH: Only perform fuzzy matching once per unique value instead of per row
    
    # Models - handle different lengths separately
    unique_models = df["model"].unique()
    model_lookup = {}
    for val in unique_models:
        if pd.isna(val) or val == "NaN":
            model_lookup[val] = "NaN"
        elif len(val) > 2:  # Only perform fuzzy matching for models that have a name longer than 2 letters -> fuzzy will become fuzzy (unreliable) if names are to short
            model_lookup[val] = process.extractOne(val, models)[0]  # [0] because we get the name and score as a return -> score used for debugging
        elif len(val) == 2:  # Use the short names list for comparisons if the model names are 2 letters
            model_lookup[val] = process.extractOne(val, short_models)[0]
        else:  # We can define models with only one letter
            model_lookup[val] = "NaN"
    df["model"] = df["model"].map(model_lookup)
    
    # Transmission
    unique_trans = df["transmission"].unique()
    trans_lookup = {val: process.extractOne(val, transmission_types)[0] for val in unique_trans}
    df["transmission"] = df["transmission"].map(trans_lookup)
    
    # FuelType
    unique_fuel = df["fuelType"].unique()
    fuel_lookup = {val: process.extractOne(val, fuel_types)[0] for val in unique_fuel}
    df["fuelType"] = df["fuelType"].map(fuel_lookup)
    
    # Convert the str NaN values back to pd.NA for easier further processing and readability
    df["model"] = df["model"].replace("NaN", pd.NA)
    df["transmission"] = df["transmission"].replace(["unkown", "NaN", "other"], pd.NA)
    df["fuelType"] = df["fuelType"].replace(["other", "NaN"], pd.NA)
    
    # Get the most frequent brand for each model -> returns df with model and brand
    brand_models = df.groupby("model")["Brand"].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else pd.NA)
    df = pd.merge(df, brand_models, on="model", how="left", suffixes=('', '_mode'))  # add the model and brand df to our main df (onyl add the brand columns, join on model)
    
    df["Brand"] = df["Brand"].fillna(df['Brand_mode'])  # rename new column
    df.drop('Brand_mode', axis=1, inplace=True)  # remove the old brand column
    
    # Cleaning numeric columns
    df['year'] = df['year'].round(0)
    
    # Create the new column
    df["stated_no_damage"] = ~df["hasDamage"].astype(bool)
    df = df.drop(["hasDamage"], axis=1)
    
    return df

In [ ]:
def fit_categorical_imputation_rules(df):
    """
    Learn conditional mode rules for categorical imputation from training data.
    
    This approach uses domain knowledge about relationships between features:
    - Brand can be inferred from model (e.g., "Golf" -> "VW")
    - Transmission preference varies by brand (e.g., BMW -> more automatic)
    - Fuel type varies by brand and year (e.g., newer cars -> more hybrid/electric)
    
    Parameters:
    -----------
    df : pd.DataFrame
        Training dataframe with string-cleaned columns
        
    Returns:
    --------
    rules : dict
        Dictionary containing conditional mode rules for each categorical feature
        
    Notes:
    ------
    This is better than simple global mode because it respects relationships:
    - "VW Golf" is more likely manual transmission
    - "BMW 5 Series" is more likely automatic transmission
    Using these relationships gives more accurate imputations than global mode.
    """
    
    rules = {}
    
    # ------------------------------------------------------------------------
    # BRAND IMPUTATION RULES
    # ------------------------------------------------------------------------
    
    # Rule: Infer brand from model (each model belongs primarily to one brand)
    # E.g., "Golf" -> "vw" (99% of Golfs are VW)
    df_with_model = df[df['model'].notna()] # Only build mapping for rows with valid model values
    rules['brand_from_model'] = df_with_model.groupby("model")["Brand"].agg(
        lambda x: x.mode()[0] if len(x.mode()) > 0 else None
    ).to_dict()
    
    # Fallback: If model is also missing, use global most common brand
    rules['brand_mode'] = df['Brand'].mode()[0] if len(df['Brand'].mode()) > 0 else 'vw'

    # ------------------------------------------------------------------------
    # MODEL IMPUTATION RULES
    # ------------------------------------------------------------------------
    
    # Rule: Infer Model from Brands
    # E.g., VW -> Golf
   
    df_with_brand = df[df['Brand'].notna()]  # Only build mapping for rows with valid brand values
    rules['model_by_brand'] = df_with_brand.groupby("Brand")["model"].agg(
        lambda x: x.mode()[0] if len(x.mode()) > 0 else None
    ).to_dict()
    
    # Fallback: Global most common transmission type
    rules['model_mode'] = df['model'].mode()[0] if len(df['model'].mode()) > 0 else 'focus'

    
    # ------------------------------------------------------------------------
    # TRANSMISSION IMPUTATION RULES
    # ------------------------------------------------------------------------
    
    # Rule: Different brands have different transmission preferences
    # E.g., BMW -> more automatic, Skoda -> more manual
   
    df_with_brand = df[df['Brand'].notna()]  # Only build mapping for rows with valid brand values
    rules['transmission_by_brand'] = df_with_brand.groupby("Brand")["transmission"].agg(
        lambda x: x.mode()[0] if len(x.mode()) > 0 else None
    ).to_dict()
    
    # Fallback: Global most common transmission type
    rules['transmission_mode'] = df['transmission'].mode()[0] if len(df['transmission'].mode()) > 0 else 'manual'
    
    # ------------------------------------------------------------------------
    # FUEL TYPE IMPUTATION RULES
    # ------------------------------------------------------------------------
    
    # Rule: Fuel type varies by brand AND year (newer cars -> more hybrid/electric)
    # Create year bins to capture this temporal trend
    df_with_brand = df[df["fuelType"].notna()] # Only build mapping for rows with valid brand values
    rules['fueltype_by_brand'] = df_with_brand.groupby("Brand")["fuelType"].agg(
        lambda x: x.mode()[0] if len(x.mode()) > 0 else None
    ).to_dict()
    
    # Fallback: Global most common fuel type
    rules['fueltype_mode'] = df['fuelType'].mode()[0] if len(df['fuelType'].mode()) > 0 else 'petrol'
    
    return rules

In [199]:
def apply_categorical_imputation(df, rules):
    """
    Apply learned categorical imputation rules to fill missing values.
    
    Uses a cascading fallback strategy:
    1. Try conditional rule (e.g., brand-specific mode)
    2. If no rule exists, use global mode
    3. Ensures no missing values remain
    
    Parameters:
    -----------
    df : pd.DataFrame
        Dataframe with missing categorical values
    rules : dict
        Dictionary of imputation rules from fit_categorical_imputation_rules()
        
    Returns:
    --------
    df : pd.DataFrame
        Dataframe with categorical missing values filled
        
    Notes:
    ------
    The cascading fallback ensures robustness:
    - If test set has a brand not seen in training, falls back to global mode
    - If a (brand, year) combination is rare, falls back to brand-only mode
    """
    
    df = df.copy()
    
    # ------------------------------------------------------------------------
    # BRAND IMPUTATION
    # ------------------------------------------------------------------------
    
    # Fill missing brands using model information with fallback to global mode
    df['Brand'] = df.apply(
        lambda row: rules['brand_from_model'].get(row['model'], rules['brand_mode']) 
                    if pd.isna(row['Brand']) else row['Brand'],
        axis=1
    )

    # ------------------------------------------------------------------------
    # MODEL IMPUTATION
    # ------------------------------------------------------------------------
    
    # Fill missing brands using model information with fallback to global mode
    df['model'] = df.apply(
        lambda row: rules['model_by_brand'].get(row['Brand'], rules['model_mode']) 
                    if pd.isna(row['model']) else row['model'],
        axis=1
    )
    
    # ------------------------------------------------------------------------
    # TRANSMISSION IMPUTATION
    # ------------------------------------------------------------------------
    
    # Fill missing transmission using brand-specific mode with fallback to global mode
    df['transmission'] = df.apply(
        lambda row: rules['transmission_by_brand'].get(row['Brand'], rules['transmission_mode'])
                    if pd.isna(row['transmission']) else row['transmission'],
        axis=1
    )
    
    # ------------------------------------------------------------------------
    # FUEL TYPE IMPUTATION
    # ------------------------------------------------------------------------
    
    # Fill missing fuel type using brand-specific mode with fallback to global mode
    df['fuelType'] = df.apply(
    lambda row: rules['fueltype_by_brand'].get(row['Brand'], rules['fueltype_mode'])
                if pd.isna(row['fuelType']) else row['fuelType'],
    axis=1
    )

    # This catches any edge cases that slipped through
    df['Brand'] = df['Brand'].fillna(rules['brand_mode'])
    df['transmission'] = df['transmission'].fillna(rules['transmission_mode'])
    df['fuelType'] = df['fuelType'].fillna(rules['fueltype_mode'])

    # Model will be imputed in numerical step, but needs a placeholder for encoding
    #df['model'] = df['model'].fillna('UNKNOWN_MODEL')
    df['model'] = df['model'].fillna(rules['model_mode'])
    
    return df

In [189]:
# ============================================================================
# SECTION 3: LABEL ENCODING (FIT ON TRAIN, TRANSFORM BOTH)
# ============================================================================

def fit_label_encoders(df):
    """
    Fit label encoders for categorical columns on training data.
    
    Label encoding converts categories to integers for use in numerical imputation.
    CRITICAL: Must be fit ONLY on training data to prevent data leakage.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Training dataframe with categorical columns
        
    Returns:
    --------
    encoders : dict
        Dictionary of fitted LabelEncoder objects for each categorical column
        
    Notes:
    ------
    We fit encoders AFTER categorical imputation so there are no missing values.
    This ensures the encoders learn a complete mapping without needing special
    handling for missing values.
    """
    
    encoders = {
        'brand': LabelEncoder(),
        'model': LabelEncoder(),
        'transmission': LabelEncoder(),
        'fuelType': LabelEncoder()
    }
    
    # Fit each encoder on the corresponding column
    encoders['brand'].fit(df['Brand'])
    encoders['model'].fit(df['model'])
    encoders['transmission'].fit(df['transmission'])
    encoders['fuelType'].fit(df['fuelType'])
    
    return encoders

In [190]:
def apply_label_encoding(df, encoders):
    """
    Apply fitted label encoders to transform categorical columns to integers.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Dataframe to encode
    encoders : dict
        Dictionary of fitted LabelEncoder objects
        
    Returns:
    --------
    df : pd.DataFrame
        Dataframe with additional encoded columns (originals preserved)
        
    Notes:
    ------
    We create NEW columns (*_encoded) rather than replacing original columns.
    This allows us to:
    1. Use encoded versions for numerical imputation
    2. Decode back to original categories after imputation
    3. Keep original categorical columns in final output
    
    If test set contains categories not seen in training, sklearn will raise
    an error. This is CORRECT behavior - we shouldn't silently make up encodings
    for unseen categories. Handle this by ensuring categorical imputation uses
    only categories seen in training.
    """
    
    df = df.copy()
    
    df['brand_encoded'] = encoders['brand'].transform(df['Brand'])
    df['model_encoded'] = encoders['model'].transform(df['model'])
    df['transmission_encoded'] = encoders['transmission'].transform(df['transmission'])
    df['fuelType_encoded'] = encoders['fuelType'].transform(df['fuelType'])
    
    return df

In [191]:
def decode_label_encoding(df, encoders):
    """
    Convert encoded integer columns back to original categorical values.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Dataframe with encoded columns (after imputation)
    encoders : dict
        Dictionary of fitted LabelEncoder objects
        
    Returns:
    --------
    df : pd.DataFrame
        Dataframe with decoded categorical columns (encoded columns removed)
        
    Notes:
    ------
    After numerical imputation, encoded values might be:
    1. Floats (from imputation algorithm)
    2. Out of valid range (e.g., negative or > max class)
    
    We handle this by:
    1. Rounding to nearest integer
    2. Clipping to valid range [0, n_classes-1]
    3. Then inverse transforming to get category names
    """
    
    df = df.copy()
    
    # Decode Brand
    n_brand_classes = len(encoders['brand'].classes_)
    brand_clipped = np.clip(df['brand_encoded'].round(), 0, n_brand_classes - 1).astype(int)
    df['Brand'] = encoders['brand'].inverse_transform(brand_clipped)
    
    # Decode Model
    n_model_classes = len(encoders['model'].classes_)
    model_clipped = np.clip(df['model_encoded'].round(), 0, n_model_classes - 1).astype(int)
    df['model'] = encoders['model'].inverse_transform(model_clipped)
    
    # Decode Transmission
    n_transmission_classes = len(encoders['transmission'].classes_)
    transmission_clipped = np.clip(df['transmission_encoded'].round(), 0, n_transmission_classes - 1).astype(int)
    df['transmission'] = encoders['transmission'].inverse_transform(transmission_clipped)
    
    # Decode FuelType
    n_fuelType_classes = len(encoders['fuelType'].classes_)
    fuelType_clipped = np.clip(df['fuelType_encoded'].round(), 0, n_fuelType_classes - 1).astype(int)
    df['fuelType'] = encoders['fuelType'].inverse_transform(fuelType_clipped)
    
    # Remove encoded columns (no longer needed)
    df.drop(["brand_encoded", "model_encoded", "transmission_encoded", "fuelType_encoded"], 
            axis=1, inplace=True)
    
    return df

#### ============================================================================
#### SECTION 4: NUMERICAL IMPUTATION (FIT ON TRAIN, TRANSFORM BOTH)
#### ============================================================================

In [ ]:
def fit_numerical_imputers(df, fast=True):
    """
    Fit imputers for numerical features on training data.
    
    Uses IterativeImputer which models each feature with missing values as a
    function of other features in a round-robin fashion. This captures
    complex relationships (e.g., year ↔ mileage ↔ price).
    
    Parameters:
    -----------
    df : pd.DataFrame
        Training dataframe with encoded categorical columns
    fast : bool
        If True, uses BayesianRidge (fast). If False, uses RandomForest (slower but better)
        
    Returns:
    --------
    imputers : dict
        Dictionary containing fitted imputer objects
        
    Notes:
    ------
    Why IterativeImputer?
    - Simple mean imputation ignores relationships (e.g., older cars have higher mileage)
    - KNN can work but doesn't model complex non-linear relationships well
    - IterativeImputer models each feature using all others, capturing correlations
    
    The estimator choice:
    - BayesianRidge: Fast, linear, works well for ~80% of cases (default sklearn)
    - RandomForest: Slower but captures non-linear relationships better
    
    We use ALL features (encoded categoricals + numericals) because:
    - Brand affects typical mileage (e.g., Mercedes often lower mileage)
    - Model affects engine size (e.g., "M3" -> large engine)
    - More features -> better imputation
    """
    
    # Define feature groups
    numerical_features = ["year", "mileage", "tax", "mpg", "engineSize", "paintQuality%", "previousOwners"]
    categorical_features_encoded = ["brand_encoded", "model_encoded", "transmission_encoded", "fuelType_encoded"]
    all_features = categorical_features_encoded + numerical_features
    
    # Select estimator based on speed/accuracy tradeoff
    if fast:
        # Use default BayesianRidge (fast, ~1 second)
        estimator = None
    else:
        # Use Random Forest for better accuracy with complex relationships (~2 minutes)
        estimator = RandomForestRegressor(
            n_estimators=20,      # Limited trees for speed
            max_depth=10,         # Prevent overfitting
            random_state=12       # Reproducibility
        )
    
    # Initialize imputer
    numerical_imputer = IterativeImputer(
        estimator=estimator,
        max_iter=10,                    # Number of imputation rounds
        random_state=12,                # For reproducibility
        initial_strategy="mean"         # Initial fill before iterative process
    )
    
    # FIT on training data
    # CRITICAL: We fit on data that still has missing values!
    # The imputer learns patterns of missingness and relationships
    numerical_imputer.fit(df[all_features])
    
    return {
        'numerical': numerical_imputer,
        'feature_names': all_features,
        'numerical_feature_names': numerical_features,
        'n_categorical': len(categorical_features_encoded)
    }


In [193]:
def apply_numerical_imputation(df, imputers):
    """
    Apply fitted numerical imputer to fill missing values.
    
    Parameters:
    -----------
    df : pd.DataFrame
        Dataframe with encoded categorical columns and numerical missing values
    imputers : dict
        Dictionary containing fitted imputer and feature information
        
    Returns:
    --------
    df : pd.DataFrame
        Dataframe with numerical missing values filled
        
    Notes:
    ------
    The imputer returns ALL features (categoricals + numericals) after imputation.
    We only extract the numerical features since:
    1. Categoricals were already imputed in Section 2
    2. Imputing encoded categoricals with regression doesn't make sense
       (it treats them as ordinal when they're nominal)
    """

    # Previous Version
    """

    print("Columns before numerical imputation:")
    print(df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    
    # Transform using fitted imputer (returns all features)
    all_imputed = imputers['numerical'].transform(df[imputers['feature_names']])
    
    # Extract only the numerical features (skip the encoded categoricals)
    n_categorical = imputers['n_categorical']
    df[imputers['numerical_feature_names']] = all_imputed[:, n_categorical:]
    """

    df = df.copy()
    
    # Ensure we're only working with the columns that were used during fit
    feature_cols = imputers['feature_names']
    
    # DEBUG: Check if all columns exist and are numeric
    missing_cols = [col for col in feature_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns for imputation: {missing_cols}")
    
    # Select only the columns used during fit (all should be numeric)
    df_for_imputation = df[feature_cols]
    
    # Verify all columns are numeric
    non_numeric = df_for_imputation.select_dtypes(exclude=['number']).columns.tolist()
    if non_numeric:
        raise ValueError(f"Non-numeric columns found: {non_numeric}. These should be encoded first!")
    
    # Transform using fitted imputer
    all_imputed = imputers['numerical'].transform(df_for_imputation)
    
    # Extract only the numerical features (skip the encoded categoricals at the start)
    n_categorical = imputers['n_categorical']
    df[imputers['feature_names']] = all_imputed 
    
    return df

#### ============================================================================
#### SECTION 5: FINAL NUMERIC CLEANUP
#### ============================================================================

In [ ]:
def final_numeric_cleanup(df):
    """
    Apply final rounding, absolute values, and domain-specific corrections.
    
    After imputation, values may be:
    - Floats that should be integers (e.g., year = 2015.3)
    - Negative when they should be positive (e.g., mileage = -1000)
    - Outside reasonable ranges due to imputation artifacts
    
    Parameters:
    -----------
    df : pd.DataFrame
        Dataframe after all imputation
        
    Returns:
    --------
    df : pd.DataFrame
        Dataframe with cleaned numerical values
        
    Notes:
    ------
    These corrections are applied identically to train and test (no fitting),
    so there's no data leakage risk. They encode domain knowledge:
    - Years are integers
    - Mileage cannot be negative
    - Paint quality is a percentage (0-100)
    - Previous owners is a count (integer ≥ 0)
    """
    
    df = df.copy()
    
    # Round year to integer (no fractional years)
    df['year'] = df['year'].round()
    
    # Mileage: take absolute value and round
    # Some imputation might produce small negative values
    df['mileage'] = abs(df['mileage'].round())
    
    # Tax: take absolute value and round
    df['tax'] = abs(df['tax'].round())
    
    # MPG: round to 1 decimal place 
    df['mpg'] = abs(df['mpg'].round(1))
    
    # Engine size: round to 1 decimal place
    df['engineSize'] = abs(df['engineSize'].round(1))
    
    # Paint quality correction (domain-specific business logic)
    # Assumption based on data exploration:
    # - Values < 4 likely had decimal point in wrong place (e.g., 3.5 -> 35%)
    # - Values > 100 likely have erroneous leading 1 (e.g., 185 -> 85%)
    def fix_paint_quality(x):
        if x < 4:
            return x * 10
        elif x > 100:
            return x - 100
        else:
            return x
    
    df["paintQuality%"] = df["paintQuality%"].apply(fix_paint_quality).round()
    
    # Previous owners: take absolute value and round to integer
    df['previousOwners'] = abs(df['previousOwners'].round())
    
    return df


#### ============================================================================
#### SECTION 6: MAIN PIPELINE FUNCTION
#### ============================================================================

In [ ]:
# Alternative way to call the pre-processing workflow if not with the sklearn class
#def clean_car_data(df_train, df_test, fast=True):
"""
Complete preprocessing pipeline for car price data.

Applies all preprocessing steps in the correct order to prevent data leakage:
1. String cleaning (no fitting required)
2. Categorical imputation (fit on train, apply to both)
3. Label encoding (fit on train, apply to both)
4. Numerical imputation (fit on train, apply to both)
5. Decode back to categories
6. Final cleanup

Parameters:
-----------
df_train : pd.DataFrame
    Training dataset with missing values and typos
df_test : pd.DataFrame
    Test dataset with missing values and typos
fast : bool, default=True
    If True, uses BayesianRidge for numerical imputation (fast ~1s)
    If False, uses RandomForest for numerical imputation (slower ~2min, slightly better)
    
Returns:
--------
df_train_cleaned : pd.DataFrame
    Fully preprocessed training data ready for modeling
df_test_cleaned : pd.DataFrame
    Fully preprocessed test data ready for predictions
    
Notes:
------
DATA LEAKAGE PREVENTION:
- All fitting (encoders, imputers, rules) happens ONLY on training data
- Test data only uses pre-fitted objects for transformation
- No information from test set influences training preprocessing

CONSISTENCY:
- Both train and test go through identical preprocessing steps
- Both are imputed (not train=dropna, test=impute)
- This ensures model sees similar distributions at train and test time

IMPUTATION STRATEGY JUSTIFICATION:
- Training on complete cases only introduces selection bias (data not MCAR)
- Train-test distribution mismatch hurts performance more than training on imputed data
- Imputing both sets with train-fitted imputers is best practice
- Literature: Saar-Tsechansky & Provost (2007), Little & Rubin (2019)
"""

# ------------------------------------------------------------------------
# STEP 1: STRING CLEANING (NO FITTING REQUIRED)
# ------------------------------------------------------------------------
print("Step 1/6: String cleaning and fuzzy matching...")
df_train_clean = process_dataframe(df_train)
df_test_clean = process_dataframe(df_test)

# ------------------------------------------------------------------------
# STEP 2: CATEGORICAL IMPUTATION (FIT ON TRAIN)
# ------------------------------------------------------------------------
print("Step 2/6: Categorical imputation (mode-based)...")

# Fit imputation rules on training data
cat_imputation_rules = fit_categorical_imputation_rules(df_train_clean)

# Apply to both datasets
df_train_cat_imputed = apply_categorical_imputation(df_train_clean, cat_imputation_rules)
df_test_cat_imputed = apply_categorical_imputation(df_test_clean, cat_imputation_rules)

# ------------------------------------------------------------------------
# STEP 3: LABEL ENCODING (FIT ON TRAIN)
# ------------------------------------------------------------------------
print("Step 3/6: Label encoding categorical features...")

# Fit encoders on training data
encoders = fit_label_encoders(df_train_cat_imputed)

# Apply to both datasets
df_train_encoded = apply_label_encoding(df_train_cat_imputed, encoders)
df_test_encoded = apply_label_encoding(df_test_cat_imputed, encoders)

# ------------------------------------------------------------------------
# STEP 4: NUMERICAL IMPUTATION (FIT ON TRAIN)
# ------------------------------------------------------------------------
print("Step 4/6: Numerical imputation (iterative)...")

# Fit imputers on training data
numerical_imputers = fit_numerical_imputers(df_train_encoded, fast=fast)

# Apply to both datasets
df_train_num_imputed = apply_numerical_imputation(df_train_encoded, numerical_imputers)
df_test_num_imputed = apply_numerical_imputation(df_test_encoded, numerical_imputers)

# ------------------------------------------------------------------------
# STEP 5: DECODE BACK TO CATEGORICAL
# ------------------------------------------------------------------------
print("Step 5/6: Decoding categorical features...")

df_train_decoded = decode_label_encoding(df_train_num_imputed, encoders)
df_test_decoded = decode_label_encoding(df_test_num_imputed, encoders)

# ------------------------------------------------------------------------
# STEP 6: FINAL CLEANUP
# ------------------------------------------------------------------------
print("Step 6/6: Final numeric cleanup...")

df_train_final = final_numeric_cleanup(df_train_decoded)
df_test_final = final_numeric_cleanup(df_test_decoded)

print("Preprocessing complete!")
print(f"Train shape: {df_train_final.shape}")
print(f"Test shape: {df_test_final.shape}")
print(f"Train missing values: {df_train_final.isna().sum().sum()}")
print(f"Test missing values: {df_test_final.isna().sum().sum()}")

return df_train_final, df_test_final"""

# Pre-processing Class for pipeline

In [196]:
from sklearn.base import BaseEstimator, TransformerMixin

class CarDataPreprocessor(BaseEstimator, TransformerMixin):
    """
    Sklearn-compatible transformer for car data preprocessing.
    
    This wrapper allows the entire preprocessing pipeline to be used in
    sklearn pipelines and cross-validation without data leakage.
    
    
    Parameters:
    -----------
    fast : bool, default=True
        If True, uses BayesianRidge for imputation (fast)
        If False, uses RandomForest for imputation (slower but better)
        
    Attributes:
    -----------
    cat_imputation_rules_ : dict
        Fitted categorical imputation rules
    encoders_ : dict
        Fitted label encoders
    numerical_imputers_ : dict
        Fitted numerical imputers
    is_fitted_ : bool
        Whether the transformer has been fitted
    """
    
    def __init__(self, fast=True):
        """
        Initialize preprocessor.
        
        Parameters:
        -----------
        fast : bool, default=True
            Speed vs accuracy tradeoff for numerical imputation
        """
        self.fast = fast
        
    def fit(self, X, y=None):
        """
        Fit all preprocessing components on training data.
        
        Parameters:
        -----------
        X : pd.DataFrame
            Training data with missing values and typos
        y : pd.Series or None
            Target variable (not used, included for sklearn compatibility)
            
        Returns:
        --------
        self : CarDataPreprocessor
            Fitted transformer
        """
        
        # Make a copy to avoid modifying original data
        X_copy = X.copy()
        
        # Step 1: String cleaning (no fitting)
        X_clean = process_dataframe(X_copy)
        
        # Step 2: Fit categorical imputation
        self.cat_imputation_rules_ = fit_categorical_imputation_rules(X_clean)
        X_cat_imputed = apply_categorical_imputation(X_clean, self.cat_imputation_rules_)
        
        # Step 3: Fit label encoders
        self.encoders_ = fit_label_encoders(X_cat_imputed)
        X_encoded = apply_label_encoding(X_cat_imputed, self.encoders_)
        
        # Step 4: Fit numerical imputers
        self.numerical_imputers_ = fit_numerical_imputers(X_encoded, fast=self.fast)
        
        # Mark as fitted
        self.is_fitted_ = True
        
        return self
    
    def transform(self, X):
        """
        Transform data using fitted preprocessing components.
        
        Parameters:
        -----------
        X : pd.DataFrame
            Data to preprocess (train or test)
            
        Returns:
        --------
        X_transformed : pd.DataFrame
            Fully preprocessed data
        """
        
        # Check if fitted
        if not hasattr(self, 'is_fitted_') or not self.is_fitted_:
            raise ValueError("Transformer must be fitted before calling transform()")
        
        # Make a copy to avoid modifying original data
        X_copy = X.copy()
        
        # Apply all preprocessing steps using fitted components
        # String Cleaning
        X_clean = process_dataframe(X_copy)
        
        # Categorical Imputation
        X_cat_imputed = apply_categorical_imputation(X_clean, self.cat_imputation_rules_)
        # Encoding
        X_encoded = apply_label_encoding(X_cat_imputed, self.encoders_)
        # Numerical Imputation
        X_num_imputed = apply_numerical_imputation(X_encoded, self.numerical_imputers_)
        # Decoding
        X_decoded = decode_label_encoding(X_num_imputed, self.encoders_)
        
        # Fix remaining numeric values issues
        X_final = final_numeric_cleanup(X_decoded)
        
        return X_final
    
    def fit_transform(self, X, y=None):
        """
        Fit and transform in one step (more efficient).
        
        Parameters:
        -----------
        X : pd.DataFrame
            Training data
        y : pd.Series or None
            Target variable
            
        Returns:
        --------
        X_transformed : pd.DataFrame
            Preprocessed training data
        """
        return self.fit(X, y).transform(X)

In [242]:
df_train = pd.read_csv("train.csv")

df_test = pd.read_csv("test.csv")

In [ ]:
df_train_preprocessed = CarDataPreprocessor(fast=True).fit_transform(df_train)

In [218]:
models_per_brand = []
results = []

brands = df_train_preprocessed["Brand"].unique()

for brand in brands:
    print(f"Training model for brand: {brand}")

    df_brand = df_train_preprocessed[df_train_preprocessed["Brand"] == brand]
    

    y_brand = df_brand["price"]
    X_brand = df_brand.drop(columns=["price"]) 
    

    num_cols = X_brand.select_dtypes(include=["int64", "float64"]).columns.tolist()
    cat_cols = X_brand.select_dtypes(include=["object", "bool"]).columns.tolist()
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", RobustScaler(), num_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
        ]
    )
    
    # pipeline
    pipeline = Pipeline([
        ("preprocess", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_split=5,
            min_samples_leaf=2,
            max_features=0.5,
            bootstrap=False,
            random_state=42
        ))
    ])
    
    # fit
    pipeline.fit(X_brand, y_brand)
    models_per_brand.append((brand, pipeline))

    y_pred = pipeline.predict(X_brand)
    
    results.append({
        "brand": brand,
        "n_samples": len(X_brand),
        "rmse": np.sqrt(mean_squared_error(y_brand, y_pred)),
        "mae": mean_absolute_error(y_brand, y_pred),
        "r2": r2_score(y_brand, y_pred)
    })

results_df = pd.DataFrame(results)
print("\n Results per Brand:")
print(results_df)


Training model for brand: vw
Training model for brand: toyota
Training model for brand: audi
Training model for brand: ford
Training model for brand: bmw
Training model for brand: skoda
Training model for brand: opel
Training model for brand: mercedes
Training model for brand: hyundai

 Results per Brand:
      brand  n_samples         rmse         mae        r2
0        vw      10604   627.506006  365.607156  0.993416
1    toyota       4715   484.024429  274.200933  0.994191
2      audi       7460   863.791279  488.829763  0.994519
3      ford      16426   504.006250  280.883647  0.989049
4       bmw       7542  1144.453441  510.662514  0.989998
5     skoda       4382   805.111577  327.696746  0.983372
6      opel       9539   386.835979  236.039565  0.988168
7  mercedes      11904  1075.010902  545.406845  0.990775
8   hyundai       3401   476.177242  290.730389  0.993609


In [238]:
class PricePredictor:
    def __init__(self,  brand_models):
        #self.general_model = general_model
        self.brand_models = dict(brand_models) 
    
    def predict(self, X, use_brand_model=True):
        preds = []
        for idx, row in X.iterrows():
            brand = row["Brand"]
            if use_brand_model and brand in self.brand_models:
                model = self.brand_models[brand]
            #else:
            #    model = self.general_model

            preds.append(model.predict(row.to_frame().T)[0])
        
        return np.array(preds)

In [243]:
predictor = PricePredictor(brand_models=models_per_brand)
# Impute missing values for test dataframe (using imputer trained on training dataframe)
df_test_preprocessed = CarDataPreprocessor(fast=True).fit(df_train).transform(df_test)

y_pred_brand = predictor.predict(df_test_preprocessed, use_brand_model=True)

print(f"Expert Model prediction: {y_pred_brand[0]:.2f}")


Expert Model prediction: 17406.52


In [262]:
predictions_df = pd.DataFrame(y_pred_brand)

submission_df = pd.concat([df_test_preprocessed["carID"], predictions_df], axis=1)

In [ ]:
submission_df.to_csv("predictions.csv", index=False, header=["carID", "price"])